# FPMarkets — Trader Data Quality Control

Goal: clean `data/fpmarkets/trader_details.csv` so it can be trusted for the ranking/strategy-selection notebook that follows (`02_trader_ranking.ipynb`).

This notebook covers, in order:
1. Load + schema overview
2. **Quality Control** — the actual cleaning steps:
   - Replace `-` placeholders with null
   - Percent columns stored as strings (`"-99.73%"`) → parsed to floats
   - `trading_*` columns → parsed to floats (fixes a stray thousands-separator comma)
   - `instruments` JSON → parsed dict
   - `trade_statistics` JSON → currency symbol pulled out into its own key, values converted to int
3. **Exploration** — diagnostics and derived columns that build on the cleaned data
4. Export the cleaned DataFrame for downstream analysis


In [1]:
import json
import re
from pathlib import Path

import pandas as pd

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RAW_PATH = Path("../../data/fpmarkets/trader_details.csv")
OUT_PATH = Path("../data/fpmarkets_clean.csv")

df = pd.read_csv(RAW_PATH)


In [3]:
df.shape

(2127, 32)

In [4]:
df.head(3)

,name,trader_id,profile_url,page_number,return_total,return_1d,age_days,return_all_time,return_year,return_half_year,return_quarter,return_month,return_week,return_day,avg_return_weekly,avg_return_monthly,return_deviation,return_deviation_monthly,return_deviation_yearly,monthly_chart,trading_return_volatility_d,trading_recovery_factor,trading_absolute_gain,trading_downside_deviation,trading_sharpe_ratio,trading_volatility_ratio,trading_max_profit,trading_max_drawdown,leverage_chart,instruments,trade_statistics,scraped_at
0,huarun1307,10338,https://socialratings.fpglobaltrading.com/widg...,1,-99.73%,0%,221,-99.73%,-,800%,2600%,2600%,0%,0%,-16.88%,-52.26%,53.46%,39.8%,53.46%,"{""Feb'26"": ""-98.58"", ""Mar'26"": ""-99.3"", ""Apr'2...",72.38,0.0,-0.31,49.47,0.08,0.03,3000%,99.99%,"[{""value"": ""0"", ""date"": ""2026-02-02""}, {""value...","{""AUDUSDxr"": ""2"", ""BTCUSD"": ""17"", ""US100"": ""51...","{""Best trade"": ""$99.65"", ""Worst trade"": ""-$94....",2026-09-11 20:30:53
1,369 NOAH 2,9845,https://socialratings.fpglobaltrading.com/widg...,1,-99.79%,-4.55%,283,-99.79%,-,5%,2000%,2000%,110%,-4.55%,-15.35%,-49.6%,46.94%,63.39%,46.94%,"{""Jan'26"": ""-99.42"", ""Feb'26"": ""-70.69"", ""Mar'...",40.60,0.0,-0.69,53.72,0.07,0.03,2200%,99.99%,"[{""value"": ""0"", ""date"": ""2026-01-01""}, {""value...","{""BTCUSD"": ""3"", ""GBPUSDxr"": ""1"", ""US100"": ""4"",...","{""Best trade"": ""€2,387.44"", ""Worst trade"": ""-€...",2026-09-11 20:31:29
2,riotokyo,11626,https://socialratings.fpglobaltrading.com/widg...,1,-99.81%,18.75%,667,-99.81%,171.43%,1800%,1800%,111.11%,58.33%,18.75%,-6.32%,-24.78%,47.54%,11.48%,34.54%,"{""Nov'24"": ""-99.01"", ""Dec'24"": ""-97.98"", ""Jan'...",50.09,0.0,-0.20,39.80,0.09,0.05,2300%,99.99%,"[{""value"": ""0"", ""date"": ""2024-11-11""}, {""value...","{""AUDCADxr"": ""1"", ""AUDCHFxr"": ""3"", ""AUDUSDxr"":...","{""Best trade"": ""$104.70"", ""Worst trade"": ""-$13...",2026-09-11 20:31:50


## Quality Control

### 2. Replace `-` placeholders with null

Several percent columns use a literal `"-"` to mean "no data for this window", not a negative sign. That's the whole meaning of the value — replace it with `NaN`.

In [5]:
# Percent-style columns that may contain a literal "-" placeholder
PERCENT_COLUMNS = [
    "return_total", "return_1d", "return_all_time", "return_year",
    "return_half_year", "return_quarter", "return_month", "return_week",
    "return_day", "avg_return_weekly", "avg_return_monthly",
    "return_deviation", "return_deviation_monthly", "return_deviation_yearly",
    "trading_max_profit", "trading_max_drawdown",
]

df[PERCENT_COLUMNS] = df[PERCENT_COLUMNS].replace("-", pd.NA)


### 3. Percent columns are strings

Every column in `PERCENT_COLUMNS` is stored as `"-99.73%"`-style text (the `-` placeholders are already `NaN` from the previous step). Strip the `%` and cast to float, passing `NaN` straight through.

In [6]:
def parse_percent(value) -> float:
    """'-99.73%' -> -99.73, NaN -> NaN."""
    if pd.isna(value):
        return float("nan")
    return float(str(value).rstrip("%"))


for col in PERCENT_COLUMNS:
    df[col] = df[col].apply(parse_percent)

df[PERCENT_COLUMNS].describe()


,return_total,return_1d,return_all_time,return_year,return_half_year,return_quarter,return_month,return_week,return_day,avg_return_weekly,avg_return_monthly,return_deviation,return_deviation_monthly,return_deviation_yearly,trading_max_profit,trading_max_drawdown
count,1807.000000,1807.000000,1803.000000,409.000000,919.00000,1521.000000,1721.000000,1792.000000,1803.000000,2122.000000,2122.000000,2122.000000,2122.000000,2122.000000,2127.000000,2127.000000
mean,75.117316,0.332037,75.312013,30.943790,1.04235,-0.511552,-0.253655,1.340826,0.332019,-7.129505,-17.902780,26.339991,9.789392,26.034651,1935.464457,54.187287
std,2804.406195,7.087041,2807.510837,282.734117,202.36950,116.475333,101.048879,61.080777,7.094537,25.422473,39.250759,51.055154,46.239808,51.135888,17053.864581,39.909839
min,-99.990000,-83.330000,-99.990000,-99.990000,-99.99000,-99.990000,-99.990000,-99.990000,-83.330000,-97.000000,-99.990000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-98.610000,0.000000,-98.610000,-58.850000,-57.67000,0.000000,0.000000,0.000000,0.000000,-11.397500,-38.882500,1.180000,0.000000,0.962500,13.085000,13.430000
50%,-35.810000,0.000000,-35.810000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,-0.440000,-1.820000,6.265000,0.000000,6.010000,95.000000,56.780000
75%,8.310000,0.000000,8.310000,27.050000,2.62500,0.000000,0.000000,0.000000,0.000000,0.020000,0.060000,33.770000,1.217500,33.440000,394.275000,98.695000
max,118103.000000,220.000000,118103.000000,4400.000000,4400.00000,2600.000000,2600.000000,1600.000000,220.000000,941.290000,941.290000,1102.690000,1102.690000,1102.690000,498858.330000,99.990000


### 4. `trading_*` columns → floats

Convert `trading_return_volatility_d`, `trading_recovery_factor`,
`trading_absolute_gain`, `trading_downside_deviation`, `trading_sharpe_ratio`,
and `trading_volatility_ratio` to float. `trading_return_volatility_d` is
otherwise plain numeric text, but one row (`MysticClimbGazelle`) has a
thousands-separator comma (`"1,763.77"`) that breaks a direct
`pd.to_numeric` cast — strip commas from all of them before casting, which is
harmless for the columns that don't need it.

In [7]:
TRADING_COLUMNS = [
    "trading_return_volatility_d",
    "trading_recovery_factor",
    "trading_absolute_gain",
    "trading_downside_deviation",
    "trading_sharpe_ratio",
    "trading_volatility_ratio",
]

for col in TRADING_COLUMNS:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(",", "", regex=False), errors="raise"
    )

df[TRADING_COLUMNS].describe()


,trading_return_volatility_d,trading_recovery_factor,trading_absolute_gain,trading_downside_deviation,trading_sharpe_ratio,trading_volatility_ratio
count,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000
mean,20.640423,0.771899,0.024673,19.884020,0.011180,0.022468
std,52.429296,9.638683,0.981154,20.964393,0.136664,0.339080
min,0.000000,0.000000,-1.020000,0.000000,-0.840000,-0.360000
25%,1.180000,0.000000,-0.270000,0.000000,-0.050000,-0.010000
50%,5.660000,0.000000,0.000000,11.850000,0.000000,0.000000
75%,22.455000,0.210000,0.100000,36.730000,0.070000,0.010000
max,1763.770000,362.360000,22.920000,99.910000,1.800000,14.690000


### 5. `instruments` → parsed dict

`instruments` is a JSON object mapping instrument symbol → trade count as a
string, e.g. `{"BTCUSD": "17", "XAUUSDxr": "1452"}`. Strip any `","`
thousands separators from the values, then cast to `int`. Rows with a missing
`instruments` value become an empty dict.

In [8]:
def parse_instruments(value) -> dict:
    if pd.isna(value):
        return {}
    raw = json.loads(value)
    return {symbol: int(count.replace(",", "")) for symbol, count in raw.items()}


df["instruments_parsed"] = df["instruments"].apply(parse_instruments)
df[["name", "instruments_parsed"]].head(5)


,name,instruments_parsed
0,huarun1307,"{'AUDUSDxr': 2, 'BTCUSD': 17, 'US100': 51, 'WT..."
1,369 NOAH 2,"{'BTCUSD': 3, 'GBPUSDxr': 1, 'US100': 4, 'XAGU..."
2,riotokyo,"{'AUDCADxr': 1, 'AUDCHFxr': 3, 'AUDUSDxr': 14,..."
3,Support and resistance/ B + R,{'XAUUSDxr': 365}
4,Minimum_Income,"{'BTCUSD': 3, 'ETHUSD': 4, 'EURUSD': 8, 'XAUUS..."


### 6. `trade_statistics` → remove currency symbol

`trade_statistics` is a JSON object like:

```json
{"Best trade": "$99.65", "Worst trade": "-$94.20", "Total profit": "-$1,395.91",
 "Trades won": "747", "Trades lost": "791", "Largest trade": "0.11",
 "Average trade size": "0.02", "Average trade duration": "00:32:28", ...}
```

⚠️ **Note**: traders report in *different account currencies* — we found `$`,
`R$` (BRL), `CA$`, `MX$`, `A$`, `ZAR`, `SGD`, `PLN`, `CHF`, `€`, `£`. The
monetary fields (`Total profit`, `Best trade`, …) aren't directly comparable
across traders until normalized to one currency — that's why we keep the
detected symbol instead of discarding it.

For every field except `Average trade duration` (a `HH:MM:SS` string, not a
number): pull the currency symbol out into its own `Currency` key in the same
dict, delete the `,` thousands separators from the value, and convert the
remaining string to an `int`.

In [9]:
DURATION_KEY = "Average trade duration"
AMOUNT_RE = re.compile(r"^(-?)([^0-9.]*)([0-9.]+)$")


def parse_trade_statistics(value) -> dict:
    if pd.isna(value):
        return {}
    raw = json.loads(value)
    parsed = {}
    currency = None
    for key, val in raw.items():
        if key == DURATION_KEY:
            parsed[key] = val  # HH:MM:SS string, not a currency value
            continue
        match = AMOUNT_RE.match(val.replace(",", "").strip())
        sign, symbol, number = match.groups()
        if symbol and currency is None:
            currency = symbol
        amount = round(float(number))
        parsed[key] = -amount if sign == "-" else amount
    parsed["Currency"] = currency
    return parsed


df["trade_statistics_parsed"] = df["trade_statistics"].apply(parse_trade_statistics)

detected_currencies = sorted({d["Currency"] for d in df["trade_statistics_parsed"] if d.get("Currency")})
print(detected_currencies)
df[["name", "trade_statistics_parsed"]].head(5)


['$', 'A$', 'CA$', 'CHF\xa0', 'MX$', 'PLN\xa0', 'R$', 'SGD\xa0', 'ZAR\xa0', '£', '€']


,name,trade_statistics_parsed
0,huarun1307,"{'Best trade': 100, 'Worst trade': -94, 'Large..."
1,369 NOAH 2,"{'Best trade': 2387, 'Worst trade': -2062, 'La..."
2,riotokyo,"{'Best trade': 105, 'Worst trade': -137, 'Larg..."
3,Support and resistance/ B + R,"{'Best trade': 270, 'Worst trade': -114, 'Larg..."
4,Minimum_Income,"{'Best trade': 47, 'Worst trade': -38, 'Larges..."


## Exploration

Diagnostics and derived columns that build on the cleaned data above — useful
for the ranking notebook, but not themselves part of the cleaning.

### Placeholder (`-`) diagnostics

Let's quantify how many `-` → `NaN` placeholders each column had, and confirm
the overlap with `age_days == 0` (brand-new traders with no track record),
before deciding whether to drop or just flag these rows.

In [10]:
null_counts = df[PERCENT_COLUMNS].isna().sum().sort_values(ascending=False)
print(null_counts)

new_traders = df[df["age_days"] == 0]
print(f"\nage_days == 0 rows: {len(new_traders)}")
print(f"of which return_year is null: {new_traders['return_year'].isna().sum()}")

# Flag so the ranking notebook can decide whether to exclude these
df["is_new_trader"] = df["age_days"] == 0
df["is_new_trader"].value_counts()


return_year                 1718
return_half_year            1208
return_quarter               606
return_month                 406
return_week                  335
return_all_time              324
return_day                   324
return_total                 320
return_1d                    320
avg_return_weekly              5
avg_return_monthly             5
return_deviation               5
return_deviation_monthly       5
return_deviation_yearly        5
trading_max_profit             0
trading_max_drawdown           0
dtype: int64

age_days == 0 rows: 320
of which return_year is null: 320


is_new_trader
False    1807
True      320
Name: count, dtype: int64

### Instrument trade-count summaries

We also derive two convenience columns (`num_instruments_traded`,
`total_instrument_trades`) since they fall straight out of the parsed dict and
will be useful for the ranking notebook (e.g. filtering out traders who only
ever traded one symbol).

In [11]:
df["num_instruments_traded"] = df["instruments_parsed"].apply(len)
df["total_instrument_trades"] = df["instruments_parsed"].apply(lambda d: sum(d.values()))

df[["name", "num_instruments_traded", "total_instrument_trades"]].head(5)


,name,num_instruments_traded,total_instrument_trades
0,huarun1307,6,1538
1,369 NOAH 2,5,2999
2,riotokyo,16,1006
3,Support and resistance/ B + R,1,365
4,Minimum_Income,4,472


## Export cleaned DataFrame

Drop the now-redundant raw JSON/string columns and save the cleaned frame to
`analysis/data/fpmarkets_clean.csv` for the ranking notebook.

In [12]:
RAW_JSON_COLUMNS = ["instruments", "trade_statistics"]
clean_df = df.drop(columns=RAW_JSON_COLUMNS)

# instruments_parsed / trade_statistics_parsed are dict columns; not CSV-friendly
# as-is, so emit them as JSON strings that round-trip cleanly.
for col in ["instruments_parsed", "trade_statistics_parsed"]:
    clean_df[col] = clean_df[col].apply(json.dumps)

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(OUT_PATH, index=False)
print(f"Saved {clean_df.shape[0]} rows x {clean_df.shape[1]} cols -> {OUT_PATH}")
clean_df.head(3)


Saved 2127 rows x 35 cols -> ..\data\fpmarkets_clean.csv


,name,trader_id,profile_url,page_number,return_total,return_1d,age_days,return_all_time,return_year,return_half_year,return_quarter,return_month,return_week,return_day,avg_return_weekly,avg_return_monthly,return_deviation,return_deviation_monthly,return_deviation_yearly,monthly_chart,trading_return_volatility_d,trading_recovery_factor,trading_absolute_gain,trading_downside_deviation,trading_sharpe_ratio,trading_volatility_ratio,trading_max_profit,trading_max_drawdown,leverage_chart,scraped_at,instruments_parsed,trade_statistics_parsed,is_new_trader,num_instruments_traded,total_instrument_trades
0,huarun1307,10338,https://socialratings.fpglobaltrading.com/widg...,1,-99.73,0.00,221,-99.73,NaN,800.0,2600.0,2600.00,0.00,0.00,-16.88,-52.26,53.46,39.80,53.46,"{""Feb'26"": ""-98.58"", ""Mar'26"": ""-99.3"", ""Apr'2...",72.38,0.0,-0.31,49.47,0.08,0.03,3000.0,99.99,"[{""value"": ""0"", ""date"": ""2026-02-02""}, {""value...",2026-09-11 20:30:53,"{""AUDUSDxr"": 2, ""BTCUSD"": 17, ""US100"": 51, ""WT...","{""Best trade"": 100, ""Worst trade"": -94, ""Large...",False,6,1538
1,369 NOAH 2,9845,https://socialratings.fpglobaltrading.com/widg...,1,-99.79,-4.55,283,-99.79,NaN,5.0,2000.0,2000.00,110.00,-4.55,-15.35,-49.60,46.94,63.39,46.94,"{""Jan'26"": ""-99.42"", ""Feb'26"": ""-70.69"", ""Mar'...",40.60,0.0,-0.69,53.72,0.07,0.03,2200.0,99.99,"[{""value"": ""0"", ""date"": ""2026-01-01""}, {""value...",2026-09-11 20:31:29,"{""BTCUSD"": 3, ""GBPUSDxr"": 1, ""US100"": 4, ""XAGU...","{""Best trade"": 2387, ""Worst trade"": -2062, ""La...",False,5,2999
2,riotokyo,11626,https://socialratings.fpglobaltrading.com/widg...,1,-99.81,18.75,667,-99.81,171.43,1800.0,1800.0,111.11,58.33,18.75,-6.32,-24.78,47.54,11.48,34.54,"{""Nov'24"": ""-99.01"", ""Dec'24"": ""-97.98"", ""Jan'...",50.09,0.0,-0.20,39.80,0.09,0.05,2300.0,99.99,"[{""value"": ""0"", ""date"": ""2024-11-11""}, {""value...",2026-09-11 20:31:50,"{""AUDCADxr"": 1, ""AUDCHFxr"": 3, ""AUDUSDxr"": 14,...","{""Best trade"": 105, ""Worst trade"": -137, ""Larg...",False,16,1006
